In [1]:
# Make this notebook work from either fine-tuning/ or fine-tuning/pre-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


cwd: C:\Users\okofoworola\Acme Health Demo\fine-tuning


# Lab 18 · Multi-Agent Flow (Agent Workflows)

Lab 11 gave us **one** agent that calls tools. But a real Member Services organisation is a **team**: a front-desk *triage* agent listens to the member, then routes the conversation to the right specialist — **Prescriptions** for a refill, **Coverage & Billing** for a copay question — and stitches the answers back together.

This lab builds that team as a genuine **multi-agent flow**, two ways:

1. **Live today — Connected Agents.** A triage agent fans out to specialist agents via `ConnectedAgentTool` (azure-ai-agents, already installed). Foundry plans the routing; we read the run steps to *see* the handoff.
2. **Production — Foundry portal Workflow.** The same flow, designed visually in the Foundry **Agent Flow / Workflows** canvas and invoked by name with `azure-ai-projects>=2.1.0`, streaming `workflow_action` events. Shown as a drop-in reference at the end.

*This is the orchestration layer most teams can't build alone — Foundry ships it as a managed service.*


---
## Step 0 — Enable Foundry tracing

*Wire OpenTelemetry to Application Insights so every agent call below shows up live in the Microsoft Foundry portal under **your project → Tracing**. For a multi-agent flow this is essential — the trace is how you see which specialist handled each member turn.*


In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from _tracing import enable_foundry_tracing

enable_foundry_tracing(service_name='acme-agentflow-lab')


GenAI tracing is not enabled. Set environment variable AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING=true to enable this experimental feature.


[tracing] enabled. Service name : acme-agentflow-lab
[tracing] Open Foundry portal -> your project -> Tracing tab
[tracing] (also visible under App Insights: appi-shuttervoice-dev)


True

---
## Step 1 — Connect to the Foundry Agent Service (LIVE)

Same project (`agents`) and the same managed-identity / `az login` credential used by the seeded production agents. Degrades gracefully if the SDK or the `Azure AI Developer` role isn't present.


In [3]:
import os
from _advisor import get_credential

ACCT    = os.environ.get('AZURE_RESOURCE_NAME', 'aif-shuttervoice-dev')
PROJECT = os.environ.get('AZURE_FOUNDRY_PROJECT', 'agents')
PROJECT_ENDPOINT = f'https://{ACCT}.services.ai.azure.com/api/projects/{PROJECT}'
MODEL = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')

# Track every agent we create so cleanup (Step 5) never leaks resources.
created_agent_ids: list[str] = []

agents = None
try:
    from azure.ai.agents import AgentsClient
    agents = AgentsClient(endpoint=PROJECT_ENDPOINT, credential=get_credential())
    print('Agents client ready ->', PROJECT_ENDPOINT)
except Exception as e:
    print('[skip] Foundry Agent Service unavailable:', type(e).__name__, e)


Agents client ready -> https://aif-shuttervoice-dev.services.ai.azure.com/api/projects/agents


---
## Step 2 — Create the specialist agents

Two domain experts, each with a tight scope. They are ordinary Foundry agents — what makes them a *team* is Step 3, where the triage agent connects to them.

- **Prescriptions specialist** — refills, status, mail-order.
- **Coverage & Billing specialist** — copays, formulary tiers, deductibles.

> **Why no client-side `FunctionTool`s here?** A connected sub-agent runs **server-side** inside the orchestrator's run. Client-side function tools need *your process* to submit their outputs — a signal the orchestrator run never surfaces, so the flow would hang. So inside a flow, specialists use **server-side capabilities** (instructions, `file_search`, Azure AI Search, code interpreter). Client-side function orchestration lives on a **single** agent — that's [Lab 11](11_agents_orchestration.ipynb). Here we keep specialists instruction-driven so the live flow completes.


In [4]:
rx_agent = coverage_agent = None
if agents is not None:
    try:
        rx_agent = agents.create_agent(
            model=MODEL, name='acme-rx-specialist',
            instructions=(
                'You are the Acme Prescriptions specialist. You handle refills, '
                'prescription status and mail-order. Member M-10293 has one expired '
                'prescription on file: lisinopril 10mg (0 refills left). When asked '
                'to refill, confirm the refill is submitted with a 2-day mail-order '
                'ETA. Be concise (one or two sentences). Do not answer billing '
                'questions.'))
        created_agent_ids.append(rx_agent.id)
        print('Prescriptions specialist:', rx_agent.id)

        coverage_agent = agents.create_agent(
            model=MODEL, name='acme-coverage-specialist',
            instructions=(
                'You are the Acme Coverage & Billing specialist. You answer copay, '
                'formulary-tier and deductible questions. Lisinopril 10mg is a Tier 1 '
                'generic with a $10 copay for this member. Be concise (one or two '
                'sentences). Do not refill medications.'))
        created_agent_ids.append(coverage_agent.id)
        print('Coverage specialist   :', coverage_agent.id)
    except Exception as e:
        print('[skip] could not create specialists:', type(e).__name__, e)
else:
    print('[skip] no agents client - see Step 1.')


Prescriptions specialist: asst_exOPxhjiyForoGolslqQImL1


Coverage specialist   : asst_Fh7QLKiTUmgx6UW5n1Wl9CkZ


---
## Step 3 — Create the triage agent that connects the team

`ConnectedAgentTool(id, name, description)` turns each specialist into a **tool** the triage agent can call. The `description` is the routing signal — Foundry reads it to decide which specialist a member turn belongs to. No glue code: the platform owns the fan-out.


In [5]:
orchestrator = None
if rx_agent is not None and coverage_agent is not None:
    try:
        from azure.ai.agents.models import ConnectedAgentTool

        rx_conn = ConnectedAgentTool(
            id=rx_agent.id, name='prescriptions_specialist',
            description='Refills, prescription status and mail-order for a member.')
        cov_conn = ConnectedAgentTool(
            id=coverage_agent.id, name='coverage_specialist',
            description='Copays, formulary tiers, deductibles and billing questions.')

        connected_defs = list(rx_conn.definitions) + list(cov_conn.definitions)
        orchestrator = agents.create_agent(
            model=MODEL, name='acme-triage-orchestrator',
            instructions=('You are the Acme Member Services triage agent. For each '
                          'member request, route the relevant part to the right '
                          'connected specialist (prescriptions vs coverage). A single '
                          'message may need BOTH. Combine their answers into one clear, '
                          'friendly reply for the member.'),
            tools=connected_defs)
        created_agent_ids.append(orchestrator.id)
        print('Triage orchestrator   :', orchestrator.id)
        print('Connected specialists :', 'prescriptions_specialist, coverage_specialist')
    except Exception as e:
        print('[skip] could not create orchestrator:', type(e).__name__, e)
else:
    print('[skip] specialists missing - see Step 2.')


Triage orchestrator   : asst_kbsHoY2zZt1maaMvw97GHmz1
Connected specialists : prescriptions_specialist, coverage_specialist


---
## Step 4 — Run a member turn that needs the whole team (LIVE)

One message, two specialties: *refill my expired med* (Prescriptions) **and** *what's my copay* (Coverage). The triage agent fans out to both connected agents, then merges the result. We read the **run steps** to visualise the flow — each connected-agent call is a node in the trace.


In [6]:
import time
if orchestrator is not None:
    try:
        thread = agents.threads.create()
        agents.messages.create(thread_id=thread.id, role='user', content=(
            'Hi, this is member M-10293. Please refill my expired lisinopril, '
            'and also tell me what my copay will be for it.'))
        # Poll explicitly with a hard timeout so the flow can never hang the notebook.
        run = agents.runs.create(thread_id=thread.id, agent_id=orchestrator.id)
        t0 = time.time()
        while run.status in ('queued', 'in_progress') and time.time() - t0 < 120:
            time.sleep(2)
            run = agents.runs.get(thread_id=thread.id, run_id=run.id)
        if run.status in ('queued', 'in_progress'):
            agents.runs.cancel(thread_id=thread.id, run_id=run.id)
        print('Run status:', run.status, f'({int(time.time() - t0)}s)')
        if getattr(run, 'last_error', None):
            print('last_error:', run.last_error)

        print('\n--- agent-flow trace (run steps) ---')
        for s in agents.run_steps.list(thread_id=thread.id, run_id=run.id):
            det = getattr(s, 'step_details', None)
            calls = getattr(det, 'tool_calls', None) or []
            if calls:
                for tc in calls:
                    name = getattr(getattr(tc, 'connected_agent', None), 'name', None) \
                           or getattr(getattr(tc, 'function', None), 'name', None) \
                           or getattr(tc, 'type', '?')
                    print(f'  step {getattr(s, "type", "?"):<16} -> routed to: {name}')
            else:
                print(f'  step {getattr(s, "type", "?"):<16} -> {getattr(det, "type", "?")}')

        print('\n--- member-facing answer ---')
        for m in agents.messages.list(thread_id=thread.id):
            if m.role == 'assistant':
                for c in m.content:
                    if getattr(c, 'text', None):
                        print('ASSISTANT:', c.text.value)
                break
    except Exception as e:
        print('[skip] flow run failed:', type(e).__name__, e)
else:
    print('[skip] no orchestrator - see Step 3.')


Run status: RunStatus.COMPLETED (17s)

--- agent-flow trace (run steps) ---


  step RunStepType.MESSAGE_CREATION -> message_creation
  step RunStepType.TOOL_CALLS -> routed to: connected_agent
  step RunStepType.TOOL_CALLS -> routed to: connected_agent



--- member-facing answer ---


ASSISTANT: Hello! Your refill for lisinopril 10mg has been submitted, and you can expect it to arrive via mail-order in about two days.

Additionally, your copay for lisinopril will be $10, since it falls under Tier 1 as a generic medication.

If you have any other questions or need further assistance, feel free to ask!


---
## Step 5 — Clean up the whole team

Every agent we created — triage **and** both specialists — is a persistent resource. Delete them all so only the seeded production agents remain. (We tracked each id in `created_agent_ids` precisely for this.)


In [7]:
if agents is not None and created_agent_ids:
    for aid in created_agent_ids:
        try:
            agents.delete_agent(aid)
            print('Deleted agent', aid)
        except Exception as e:
            print('[warn] cleanup', aid, ':', e)
else:
    print('Nothing to clean up.')


Deleted agent asst_exOPxhjiyForoGolslqQImL1


Deleted agent asst_Fh7QLKiTUmgx6UW5n1Wl9CkZ


Deleted agent asst_kbsHoY2zZt1maaMvw97GHmz1


---
## Step 6 — Production path: the Foundry portal Workflow (reference)

The Connected-Agents flow above is built **in code**. In production you usually design the same flow **visually** in the Foundry **Agent Flow / Workflows** canvas (drag agents, draw the routing edges, version it), then invoke it by name. That uses the `azure-ai-projects>=2.1.0` Responses API (`get_openai_client().responses.create(...)`) and streams the workflow's `workflow_action` items so you can watch each actor fire in real time.

This cell is a **drop-in reference**: design a workflow in the portal, then set `ACME_WORKFLOW_NAME` to its name (or edit below). Until a workflow name is provided it skips cleanly. Note: in newer SDKs the stream events are plain strings (e.g. `response.output_text.done`), so we match `event.type` directly rather than importing an enum.


In [8]:
# Replace-in-production: design the flow in the Foundry Workflows canvas,
# publish it, then set ACME_WORKFLOW_NAME to invoke it by reference.
WORKFLOW_NAME = os.environ.get('ACME_WORKFLOW_NAME', '')  # e.g. 'acme-member-services-flow'

try:
    from azure.ai.projects import AIProjectClient
    _has_workflows = hasattr(AIProjectClient, 'get_openai_client')
except Exception:
    _has_workflows = False

if not _has_workflows:
    print('[skip] portal Workflows need azure-ai-projects>=2.1.0 '
          '(installed build lacks get_openai_client).')
    print('       pip install "azure-ai-projects>=2.1.0"  then design a flow in the portal.')
elif not WORKFLOW_NAME:
    print('[skip] set ACME_WORKFLOW_NAME (or edit this cell) to a published workflow name.')
else:
    project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=get_credential())
    with project_client:
        oai = project_client.get_openai_client()
        conv = oai.conversations.create()
        print('conversation:', conv.id)
        stream = oai.responses.create(
            conversation=conv.id,
            extra_body={'agent_reference': {'name': WORKFLOW_NAME, 'type': 'agent_reference'}},
            input='Refill my lisinopril and tell me the copay.',
            stream=True,
            metadata={'x-ms-debug-mode-enabled': '1'},
        )
        for event in stream:
            etype = getattr(event, 'type', '')
            item = getattr(event, 'item', None)
            is_wf = getattr(item, 'type', None) == 'workflow_action'
            if etype == 'response.output_text.done':
                print('TEXT:', getattr(event, 'text', ''))
            elif etype == 'response.output_item.added' and is_wf:
                print(f"ACTOR -> {item.action_id} ({item.status})")
            elif etype == 'response.output_item.done' and is_wf:
                print(f"DONE  -> {item.action_id} ({item.status})")
        oai.conversations.delete(conversation_id=conv.id)
        print('conversation deleted')


[skip] set ACME_WORKFLOW_NAME (or edit this cell) to a published workflow name.


---
## Takeaways

- A **multi-agent flow** turns one assistant into a coordinated team: a triage agent routes each member turn to the right specialist and merges the answers.
- `ConnectedAgentTool` makes the wiring **declarative** — the specialist's `description` is the routing signal; Foundry owns the fan-out and retries.
- The **run steps** are your flow trace: every connected-agent call is a node you can replay in the Foundry **Tracing** tab (Step 0).
- For production, design the same flow **visually** in the Foundry **Workflows** canvas and invoke it by name (`azure-ai-projects>=2.1.0`, Step 6) — version-controlled, no glue code.
- Always **clean up** created agents (Step 5) so the project stays tidy.

*← The Decision Advisor (Lab 09) routes the `needs_agents` flag to the agent track: Lab 11 (single agent + tools) → Lab 18 (multi-agent flow).*
